## Problem Statement
Diabetes mellitus is a growing public health problem in Uganda and across Africa, with the World Health Organization reporting an increase in cases due to lifestyle changes and limited early screening. Late diagnosis often leads to serious complications such as kidney failure, blindness, heart disease, and amputations, putting immense pressure on patients, families, and the healthcare system.

This project addresses the real-world problem of early diabetes risk prediction using machine learning. Using patient clinical data (such as glucose levels, BMI, age, and insulin), we  built a supervised binary classification model that predicts whether a patient is likely to have diabetes (Outcome = 1) or not (Outcome = 0).


# Why this problem matters
Early detection allows for timely lifestyle interventions, dietary changes, and medical follow-up, which can significantly reduce complications and healthcare costs.


# Stakeholders

Healthcare providers and hospitals in Uganda
Patients and their families
Public health officials and the Ministry of Health

# Decisions that can be improved
A machine learning solution can directly improve critical clinical and public-health decisions by providing fast, data-driven risk scores instead of relying only on manual assessment or expensive lab tests.

# Project Objectives

Acquire, explore, and preprocess a suitable diabetes dataset.

Build, train, and tune two to three classification models to predict diabetes risk.

Evaluate the models rigorously and provide actionable insights for real-world use in Ugandan healthcare settings.

### (b) Dataset Acquisition

 The dataset used is the **Pima Indians Diabetes Dataset**.  

 It meets all requirements:  
   - 768 records (rows)  
   - 9 features (columns) with a mix of numeric and binary variables  
   - Clear target variable: `Outcome` (0 = non-diabetic, 1 = diabetic)  

 **Full citation and URL**  
Smith, J.W., Everhart, J.E., Dickson, W.C., Knowler, W.C., & Johannes, R.S. (1988). Using the ADAP learning algorithm to forecast the onset of diabetes mellitus. In Proceedings of the Symposium on Computer Applications and Medical Care (pp. 261-265). IEEE Computer Society.  

**URL**: https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database  


### Loading the dataset

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:

df = pd.read_csv("pima_diabetes_data.csv")
df.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


### Checking for a data shape of the dataset

In [3]:
df.shape

(768, 9)

# Column names

In [5]:
#column names
list(df.columns)

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age',
 'Outcome']

# Data types

In [6]:
df.dtypes

Pregnancies                   int64
Glucose                       int64
BloodPressure                 int64
SkinThickness                 int64
Insulin                       int64
BMI                         float64
DiabetesPedigreeFunction    float64
Age                           int64
Outcome                       int64
dtype: object

 **Data Dictionary**

| Column                    | Description                                           | Type      |
|---------------------------|-------------------------------------------------------|-----------|
| Pregnancies               | Number of times pregnant                              | Integer   |
| Glucose                   | Plasma glucose concentration (mg/dL)                  | Integer   |
| BloodPressure             | Diastolic blood pressure (mm Hg)                      | Integer   |
| SkinThickness             | Triceps skin fold thickness (mm)                      | Integer   |
| Insulin                   | 2-hour serum insulin (mu U/ml)                        | Integer   |
| BMI                       | Body Mass Index (kg/m²)                               | Float     |
| DiabetesPedigreeFunction  | Diabetes pedigree function (genetic risk)             | Float     |
| Age                       | Age of the patient (years)                            | Integer   |
| Outcome                   | Target: 1 = diabetic, 0 = non-diabetic                | Integer   |

# (i) Descriptive statistics for all numeric features

In [4]:
df.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


### Checking for missing values

In [7]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

### Detecting the hidden missing values

In [8]:
(df == 0).sum()

Pregnancies                 111
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                     500
dtype: int64

- Certain features such as Glucose, BloodPressure, SkinThickness, Insulin, and BMI contained zero values that are not medically valid. These values were treated as missing data and replaced with NaN to allow proper imputation during preprocessing. The target variable (Outcome) and other valid features were left unchanged.

In [9]:
cols_to_fix = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_to_fix] = df[cols_to_fix].replace(0, np.nan)

###  Handle Missing Values using Median

In [10]:
df.fillna(df.median(), inplace=True)

###  Verify Missing Values After Replacement

In [11]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

- We use median to impute because columns like insulin skin thickness are right skewed the tail moves to the right

- No categorical (string) variables in the dataset. All features are numeric.

- Therefore, no label encoding or one-hot encoding was required.

## EDA